# MNIST digit classification with and without hidden layers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tensorchiefs/dl_course_2025/blob/master/notebooks/03_fcnn_mnist_keras_torch.ipynb)

In this notebook you will use the MNIST dataset for a classification task. You will compare the performance of a fully connected neural network with and without hidden layers.


**Dataset:** You work with the MNIST dataset. We have 60'000 28x28 pixel greyscale images of handwritten digits and want to classify them into the correct label (0-9).

**Content:**
* load the original MNIST data
* visualize samples of the data
* flatten the data
* use keras to train a fcNN with and without hidden layers and compare the performance on new unseen test data

#### Imports

In the next two cells, we load all the required libraries and functions. We download the Mnist data, normalize the pixelvalues to be between 0 and 1, and separate it into a training and validation set.

In [ ]:
# load required libraries:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('default')
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import os
os.environ["KERAS_BACKEND"] = "jax"
import keras
import torch # not needed yet

print(f'Keras_version: {keras.__version__}')# 3.5.0
print(f'torch_version: {torch.__version__}')# 2.5.1+cu121
print(f'keras backend: {keras.backend.backend()}')

# Keras Building blocks
from keras.models import Sequential
from keras.layers import Dense, Convolution2D, MaxPooling2D, Flatten , Activation
from keras.optimizers import SGD, Adam
from keras.utils import to_categorical
from keras import optimizers


In [ ]:
# 🔧 Define a helper function to plot learning curves
def plot_learning_curves(history):
    fig, axs = plt.subplots(1,2, figsize=(12,4))
    ax = axs[0]
    ax.plot(history.history['loss'], label="training")
    if history.history.get('val_loss') is not None:
        ax.plot(history.history['val_loss'], label="validation")
    ax.set_title('model loss')
    ax.set_ylabel('loss')
    ax.set_xlabel('epoch')
    ax.legend()
    ax.grid()

    ax = axs[1]
    ax.plot(history.history['accuracy'], label="training")
    if history.history.get('val_accuracy') is not None:
        ax.plot(history.history['val_accuracy'], label="validation")
    ax.set_title('model accuracy')
    ax.set_ylabel('accuracy')
    ax.set_xlabel('epoch')
    ax.legend()
    ax.grid()

    return fig

# 🔧 Helper function to reset model weights
def reset_weights(model):
    for layer in model.layers:
        # Check if the layer has initializers (weights and biases)
        if hasattr(layer, 'kernel_initializer') and hasattr(layer, 'bias_initializer'):
            # Generate new weights using the layer's original initializer
            new_weights = layer.kernel_initializer(layer.kernel.shape)
            new_biases = layer.bias_initializer(layer.bias.shape)
            
            # Update the layer weights
            layer.set_weights([new_weights, new_biases])






#### Loading and preparing the MNIST data

In [ ]:
from keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# separate x_train in X_train and X_val, same for y_train
X_train=x_train[0:50000] / 255 #divide by 255 so that they are in range 0 to 1
Y_train=to_categorical(y_train[0:50000],10) # one-hot encoding

X_val=x_train[50000:60000] / 255
Y_val=to_categorical(y_train[50000:60000],10)

X_test=x_test / 255
Y_test=to_categorical(y_test,10)

del x_train, y_train, x_test, y_test

X_train=np.reshape(X_train, (X_train.shape[0],28,28,1))
X_val=np.reshape(X_val, (X_val.shape[0],28,28,1))
X_test=np.reshape(X_test, (X_test.shape[0],28,28,1))

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

Note: This type of tensor structure is typical for images in the context of deep learning.

- 1st dimension: identifies the data point in the batch
- 2nd and 3rd dimension: the 2D value grid (e.g. intensity over 2 spatial dimensions)
- 4th dimension: so called "channel", e.g. color or filter bank ID.

This way, you have a data structure that holds a set of multi-channel 2D-images.

Let's visualize the first 4 mnist images. It is very easy to recognise the true label of the digits.

In [ ]:
# visualize the 4 first mnist images
plt.figure(figsize=(12,12))
for i in range(0,4):
    plt.subplot(1,4,(i+1))
    plt.imshow((X_train[i,:,:,0]),cmap="gray")
    plt.title('true label: '+str(np.argmax(Y_train,axis=1)[i]))
    #plt.axis('off')

## fcNN as classification model for MNIST data

Now we want to train a fully connected neural network (fcNN) to classify the MNIST data.
We use two network architectures:

* fcnn with no hidden layers
* fcnn with two hidden layers (100 and 50)


Because we will use fcNN we need to flatten our input into a 1d vector. We do this in the next cell with reshape.

In [ ]:
# prepare data for fcNN - we need a vector as input

# first do it for original data
X_train_flat = X_train.reshape([X_train.shape[0], 784])
X_val_flat = X_val.reshape([X_val.shape[0], 784])
X_test_flat = X_test.reshape([X_test.shape[0], 784])

### Train single hidden layer neural network

In [ ]:
# check the shape
X_train_flat.shape,Y_train.shape,X_val_flat.shape,Y_val.shape

Here we define the network. In the output we predict the probability for the 10 digits with the softmax activation function.

In [ ]:
# Define fcNN with 1 hidden layer
# Use functional definition
# Our network starts with two inputs
inputs = keras.Input(shape=(784,))
output = Dense(10, activation="softmax")(inputs)
model_neuron = keras.Model(inputs, output)

# compile model and intitialize weights
model_neuron.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
# summarize model along with number of model weights
model_neuron.summary()

In [ ]:
# train the model
reset_weights(model_neuron)
history = model_neuron.fit(
    X_train_flat, Y_train,
    batch_size=512,
    epochs=30,
    validation_data=(X_val_flat, Y_val),
    verbose=0
)

In [ ]:
fig = plot_learning_curves(history);

#### Prediction on the test set

Now, let's use the fcNN that was trained to predict new unseen data (our testdata).
We determine the confusion matrix and the accuracy on the testdata to evaluate the classification performance.


In [ ]:
# predict each instance of the testset
pred=model_neuron.predict(X_test_flat)
# get confusion matrix
cm = confusion_matrix(np.argmax(Y_test, axis=1), np.argmax(pred, axis=1))

acc_fc = np.sum(np.argmax(Y_test,axis=1)==np.argmax(pred,axis=1))/len(pred)
print("Accuracy = " , acc_fc)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='viridis')
plt.title('Confusion Matrix')
plt.show()

### Train deeper fcNN on the data

Now, we define the nework with two hidden layers (100, 50). We use the sigmoid activation function on the hidden layers. In the output we predict the probability for the 10 digits with the softmax activation function.

In [ ]:
# Define fcNN with 2 hidden layers
inputs = keras.Input(shape=(784,))
x = Dense(100, activation='sigmoid')(inputs)
x = Dense(50, activation='sigmoid')(x)
outputs = Dense(10, activation='softmax')(x)
model_deep = keras.Model(inputs, outputs)

# Compile model and intitialize weights
model_deep.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
# summarize model along with number of model weights
model_deep.summary()

In [ ]:
# train the model
history=model_deep.fit(X_train_flat, Y_train,
                  batch_size=512,
                  epochs=50,
                  verbose=1,
                  validation_data=(X_val_flat, Y_val)
                 )

In [ ]:
plot_learning_curves(history);

🖐 **Note:** The fact that the validation loss is lower than the training loss is statistically and mathematically counter-intuitive. It is in deed an artifact of the way Keras computes the losses: The training loss is averaged over all batches of the epoch, whereas the validation loss is computed at the end of an epoch. Because the model keeps improving during the epoch, the average of training losses over the epoch can be higher than the validation loss at the end. But this will stop being the case once the model becomes better at the prediction task.

In [ ]:
# predict each instance of the testset
pred = model_deep.predict(X_test_flat)
# get confusion matrix
cm = confusion_matrix(np.argmax(Y_test, axis=1), np.argmax(pred, axis=1))

acc_fc = np.sum(np.argmax(Y_test,axis=1)==np.argmax(pred,axis=1))/len(pred)
print("Accuracy = " , acc_fc)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='viridis')
plt.title('Confusion Matrix')
plt.show()

### 🔧 **YOUR TASK:**


What do you observe when you compare the network with the hidden layers to the one without?  
  
Try to improve the fcNN by adding more hidden layers and/or changing the activation function from "sigmoid" to "relu". What do you observe?

In [ ]:
## YOUR CODE HERE ###

### 🔑 **Solution:**


>



<details>
  <summary>🔑 Click here to View Answers:</summary>



- Accuracy increases with hidden layer, from ~ 0.93 to ~ 0.97 , while the loss (NLL) decreases from ~ 0.26 to ~ 0.1

- With ReLu, faster convergence is achieved

</details>

In [ ]:
from keras.layers import Dropout

# Functional definition of a much more complex, fully connected neural network with moderate dropout
input = keras.Input(shape=(784,))
x = Dense(512, activation='relu')(input)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(10, activation='softmax')(x)
model_func = keras.Model(input, output)

# compile model and initialize weights
adam = optimizers.Adam()
model_func.compile(loss='categorical_crossentropy',
              optimizer=adam,
              metrics=['accuracy'])

In [ ]:
# Functional definition of a much more complex, fully connected neural network
input = keras.Input(shape=(784,))
x = Dense(512, activation='relu')(input)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
output = Dense(10, activation='softmax')(x)
model_deeper = keras.Model(input, output)

# compile model and initialize weights
# You could provide a learning rate to Adam, like `learning_rate=0.001`, but the adaptive 
# nature of the Adam optimizer makes this less important
adam = optimizers.Adam()
model_deeper.compile(loss='categorical_crossentropy',
              optimizer=adam,
              metrics=['accuracy'])

In [ ]:
# summarize model along with number of model weights
model_deeper.summary()

In [ ]:
# train the model
history = model_deeper.fit(X_train_flat, Y_train,
                  batch_size=512,
                  epochs=50,
                  verbose=1,
                  validation_data=(X_val_flat, Y_val)
                 )

In [ ]:
plot_learning_curves(history);

In [ ]:
# predict each instance of the testset
pred=model_deeper.predict(X_test_flat)
# get confusion matrix
cm = confusion_matrix(np.argmax(Y_test, axis=1), np.argmax(pred, axis=1))

acc_fc = np.sum(np.argmax(Y_test,axis=1)==np.argmax(pred,axis=1))/len(pred)
print("Accuracy = " , acc_fc)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='viridis')
plt.title('Confusion Matrix')
plt.show()

### ✏️ **YOUR TASK:**

You've just seen 

- a single neuron network, 
- a two hidden layer neural network and 
- an even deeper neural network

Trained and validated on the MNIST 10 digit classification task. What have your observations been regarding

- Learning curves? Train and validation.
- Test performance?


<details>
  <summary>🔑 Click here to View Answers:</summary>



- The simple, single neuron network seems to have converged to its solution after the first epoch. 
- This is due to the fact that we are dealing with a trivial network setup, where the loss landscape is convex and low-dimensional.

</details>

